# FEMI-2: Fixed-message evidence-absence pilot
This notebook loads the **locally frozen** 144-request packet. Preflight/tests never call the model; live cells are billed if enabled. Do not change prompts or stop early based on outcomes. Set `OPENROUTER_API_KEY` in Colab Secrets.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, subprocess, os
uploaded=files.upload()
path=Path(next(iter(uploaded)))
assert path.suffix=='.zip'
with zipfile.ZipFile(path) as z: z.extractall('/content')
P=Path('/content/FEMI2_FixedPilot_20261108')
assert (P/'offline/femi2_freeze.json').exists(), 'Please upload the downloaded FEMI-2 ZIP'
print('Loaded',P)


In [ ]:
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'],cwd=P,check=True)
subprocess.run(['python','verify_femi1_integrity.py','--packet','reference/FEMI1_Local_Checksum_Fix_20261107.zip','--results','reference/FEMI1_LIVE_RESULTS.zip'],cwd=P,check=True)
subprocess.run(['python','run_femi2_openrouter.py','--manifest','offline/femi2_manifest_144.jsonl','--freeze','offline/femi2_freeze.json','--out','femi2_results'],cwd=P,check=True)


## Only run the next cell to collect 144 *paid* responses
The cell uses an explicitly configured secret. It will resume safely if interrupted. If you only want to inspect the design, stop here.


In [ ]:
from google.colab import userdata
KEY=userdata.get('OPENROUTER_API_KEY')
assert KEY and KEY.strip(), 'Set OPENROUTER_API_KEY in Colab Secrets first'
env=dict(os.environ);env['OPENROUTER_API_KEY']=KEY
subprocess.run(['python','run_femi2_openrouter.py','--manifest','offline/femi2_manifest_144.jsonl','--freeze','offline/femi2_freeze.json','--out','femi2_results','--live','--pace','1.0'],cwd=P,env=env,check=True)
del KEY; env.pop('OPENROUTER_API_KEY',None)


In [ ]:
subprocess.run(['python','analyze_femi2.py','--manifest','offline/femi2_manifest_144.jsonl','--freeze','offline/femi2_freeze.json','--results','femi2_results/responses.jsonl','--out','femi2_results/analysis'],cwd=P,check=True)
import shutil
archive=shutil.make_archive('/content/FEMI2_LIVE_RESULTS','zip',root_dir=P/'femi2_results')
files.download(archive)
